# MLP Superposition Detection

Detect superposed representations in MLPs: neuron correlations,
output interference, polysemanticity, and dimensionality.

## Why This Matters

MLP superposition detection provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_superposition_detection import (
    neuron_activation_correlation, neuron_output_interference,
    neuron_polysemanticity, superposition_dimensionality,
    mlp_superposition_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Neuron Activation Correlations

High correlations between neurons suggest shared features.

In [ ]:
for layer in range(4):
    result = neuron_activation_correlation(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_off_diag={result['mean_off_diagonal']:.4f}, "
          f"max_corr={result['max_correlation']:.4f}")

## Output Direction Interference

Neuron pairs whose output directions interfere in the residual stream.

In [ ]:
for layer in range(4):
    result = neuron_output_interference(model, tokens, layer=layer, top_k=3)
    print(f"  Layer {layer}: mean_interference={result['mean_interference']:.4f}")
    for p in result['most_interfering']:
        print(f"    N{p['neuron_a']}-N{p['neuron_b']}: {p['interference']:.4f}")

## Polysemanticity

Neurons that activate uniformly across positions are more polysemantic.

In [ ]:
for layer in range(4):
    result = neuron_polysemanticity(model, tokens, layer=layer, top_k=3)
    most = [(n, f'{s:.3f}') for n, s in result['most_polysemantic']]
    print(f"  Layer {layer}: mean={result['mean_polysemanticity']:.4f}, top={most}")

## Superposition Dimensionality

In [ ]:
for layer in range(4):
    result = superposition_dimensionality(model, tokens, layer=layer)
    print(f"  Layer {layer}: eff_rank={result['effective_rank']:.2f}, "
          f"d_mlp={result['d_mlp']}, ratio={result['superposition_ratio']:.2f}x")

## Summary

In [ ]:
result = mlp_superposition_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: corr={p['mean_correlation']:.4f}, "
          f"super_ratio={p['superposition_ratio']:.2f}x")